In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

# KPI: FLEET HEALTH BY SITE (Aggregating across machines)
# 👉 FIX: 2-part 'gold.*' names ki jagah env-aware f"{GOLD}.*"
dim_machine = spark.read.table(f"{GOLD}.dim_machine").filter(col("__CURRENT") == True)
dim_site = spark.read.table(f"{GOLD}.dim_site")
mtbf = spark.read.table(f"{GOLD}.kpi_mtbf")
util = spark.read.table(f"{GOLD}.fact_utilization_daily")

fleet_health = dim_machine.join(dim_site, "site_id").join(mtbf, "machine_sk", "left").join(util, "machine_sk", "left") \
    .groupBy("site_sk", "site_name").agg(
        avg("mtbf_hours").alias("avg_mtbf"),
        avg("utilization_pct").alias("avg_utilization"),
        countDistinct("machine_sk").alias("total_machines")
    )
fleet_health.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.mart_fleet_health")

print("✅ gold_09_mart_summary built. ALL GOLD TABLES READY!")